# data/cleaned jsonl 파일 EDA

`data/cleaned/` 폴더의 8개 `*_cleaned.jsonl` 파일을 pandas로 읽어

1. 컬럼별 결측치(null) 개수를 확인하고
2. `abstract` 컬럼의 단어 수(min / max / mean tokens)를 확인합니다.

In [ ]:
import glob
import os

import pandas as pd

DATA_DIR = "../data/cleaned"

In [ ]:
jsonl_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.jsonl")))
jsonl_paths

['../data/cleaned\\arxiv_cs_recent_3months_part1_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part2_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part3_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part4_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part5_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part6_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part7_cleaned.jsonl',
 '../data/cleaned\\arxiv_cs_recent_3months_part8_cleaned.jsonl']

In [ ]:
dfs = {}
for path in jsonl_paths:
    name = os.path.basename(path)
    df = pd.read_json(path, lines=True)
    df["source_file"] = name
    dfs[name] = df
    print(f"{name}: {len(df)} rows, columns={list(df.columns[:-1])}")

combined_df = pd.concat(dfs.values(), ignore_index=True)
print(f"\ntotal: {len(combined_df)} rows")

arxiv_cs_recent_3months_part1_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']
arxiv_cs_recent_3months_part2_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']


arxiv_cs_recent_3months_part3_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']
arxiv_cs_recent_3months_part4_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']


arxiv_cs_recent_3months_part5_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']
arxiv_cs_recent_3months_part6_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']


arxiv_cs_recent_3months_part7_cleaned.jsonl: 5000 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']
arxiv_cs_recent_3months_part8_cleaned.jsonl: 3479 rows, columns=['id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published']

total: 38479 rows


## 1. 결측치 확인

In [ ]:
# 파일별 컬럼별 결측치 개수
missing_by_file = pd.DataFrame(
    {name: df.drop(columns="source_file").isnull().sum() for name, df in dfs.items()}
).T
missing_by_file

,id,title,abstract,authors,categories,primary_category,published
arxiv_cs_recent_3months_part1_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part2_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part3_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part4_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part5_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part6_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part7_cleaned.jsonl,0,0,0,0,0,0,0
arxiv_cs_recent_3months_part8_cleaned.jsonl,0,0,0,0,0,0,0


In [ ]:
# 전체 파일 합산 결측치 개수
missing_total = combined_df.drop(columns="source_file").isnull().sum()
missing_total

id                  0
title               0
abstract            0
authors             0
categories          0
primary_category    0
published           0
dtype: int64

## 2. abstract 컬럼 단어 수(token) 통계

In [ ]:
# 공백 기준 단어 수를 토큰 수로 사용
combined_df["abstract_token_count"] = combined_df["abstract"].fillna("").str.split().apply(len)

In [ ]:
# 파일별 min / max / mean 토큰 수
token_stats_by_file = combined_df.groupby("source_file")["abstract_token_count"].agg(
    min_tokens="min", max_tokens="max", mean_tokens="mean"
)
token_stats_by_file

,min_tokens,max_tokens,mean_tokens
source_file,,,
arxiv_cs_recent_3months_part1_cleaned.jsonl,25,333,189.183600
arxiv_cs_recent_3months_part2_cleaned.jsonl,24,322,188.569600
arxiv_cs_recent_3months_part3_cleaned.jsonl,16,365,185.529400
arxiv_cs_recent_3months_part4_cleaned.jsonl,15,401,186.593800
arxiv_cs_recent_3months_part5_cleaned.jsonl,20,313,189.845800
arxiv_cs_recent_3months_part6_cleaned.jsonl,20,329,190.584800
arxiv_cs_recent_3months_part7_cleaned.jsonl,20,327,192.558400
arxiv_cs_recent_3months_part8_cleaned.jsonl,14,325,191.239149


In [ ]:
# 전체 파일 통합 min / max / mean 토큰 수
overall_stats = combined_df["abstract_token_count"].agg(min_tokens="min", max_tokens="max", mean_tokens="mean")
overall_stats

min_tokens      14.000000
max_tokens     401.000000
mean_tokens    189.184958
Name: abstract_token_count, dtype: float64